# SIBI Dynamic Point-History Classifier (Phase 2F)

Trains an LSTM (following Kazuhito Takahashi's `point_history_classification` reference) that maps a 32-step wrist trajectory (32 × 2 = 64 floats) to one of the dynamic SIBI classes — word gestures (jeruk, kucing, besar, kecil, sangat) plus the motion-letters J and Z.

**Data shape**: each sample is 32 points sampled UNIFORMLY along a fixed 1.5-second window (wall-clock). The frontend recorder collects raw timestamped wrist positions over that window, then linear-interpolates them to 32 evenly-spaced output points. This decouples the trained model from the recording laptop's MediaPipe inference rate — a "jeruk" gesture has the same temporal shape whether collected on a 30fps machine or a 10fps one.

**Older CSVs collected before the time-resample migration** (raw 32 consecutive frames, variable wall-clock duration) are NOT compatible with this dataset and should be cleared / re-collected.

Input CSV: `point_history_csv/dynamic.csv` (collected via `/dev/gesture-recorder` or `frontend/public/recorder.html`).
Output: `point_history_classifier.hdf5`, `point_history_classifier.tflite`.

In [ ]:
import csv
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
# 5 word gestures + 2 motion-letters (J, Z) = 7 dynamic classes.
NUM_CLASSES = 7
FEATURE_LENGTH = 64  # 32 frames * 2 coords (must match DYNAMIC_HISTORY_SIZE in frontend recording/types.ts)
CSV_PATH = 'point_history_csv/dynamic.csv'
LABEL_CSV_PATH = 'point_history_csv/dynamic_label.csv'
TFLITE_PATH = 'point_history_classifier.tflite'
HDF5_PATH = 'point_history_classifier.hdf5'

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Total samples: {len(df)}')
print(f'Samples per class:\n{df["label"].value_counts()}')

labels = pd.read_csv(LABEL_CSV_PATH, header=None)[0].tolist()
label_to_idx = {l: i for i, l in enumerate(labels)}
assert len(labels) == NUM_CLASSES

X = df.iloc[:, 1:].astype(np.float32).values
y_strings = df['label'].astype(str).values
y = np.array([label_to_idx[l] for l in y_strings], dtype=np.int32)
assert X.shape[1] == FEATURE_LENGTH
print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.75, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# LSTM variant from Kazuhito's point_history_classification.ipynb.
# Reshape the flat (TIME_STEPS * DIMENSION,) feature vector into a proper
# (TIME_STEPS, DIMENSION) sequence so the LSTM can capture temporal order —
# critical for motion words like jeruk/kucing vs. their reverse-direction twins.
TIME_STEPS = 32  # = DYNAMIC_HISTORY_SIZE
DIMENSION = 2  # (x, y) per frame
assert TIME_STEPS * DIMENSION == FEATURE_LENGTH

model = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(input_shape=(FEATURE_LENGTH,)),
    tf.keras.layers.Reshape((TIME_STEPS, DIMENSION), input_shape=(FEATURE_LENGTH,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(16, input_shape=[TIME_STEPS, DIMENSION]),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    HDF5_PATH, verbose=1, save_weights_only=False, save_best_only=True,
)
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

history = model.fit(
    X_train, y_train,
    epochs=1000,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback],
    verbose=2,
)

In [ ]:
model = tf.keras.models.load_model(HDF5_PATH)
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(y_test, y_pred, target_names=labels, digits=3))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('SIBI Dynamic Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized = converter.convert()
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_quantized)
print(f'Wrote {TFLITE_PATH} ({len(tflite_quantized)} bytes)')